<img src="https://hilpisch.com/tds_logo.png" width="20%" align="right">


### Why this notebook exists
The book needs a bridge between first-table habits and the more realistic
workflows that use multiple files. This notebook gives that bridge a small,
repeatable shape.


This cell reads the dataset into a DataFrame so the later examples can inspect and transform it.


In [ ]:
from pathlib import Path

import pandas as pd

data_dir = Path('data')
artifacts_dir = Path('artifacts')
artifacts_dir.mkdir(parents=True, exist_ok=True)

orders = pd.read_csv(data_dir / 'orders.csv')
customers = pd.read_csv(data_dir / 'customers.csv')

print(orders.head())
print()
print(customers.head())


This cell continues the worked example for the current section.


In [ ]:
orders['order_date'] = pd.to_datetime(orders['order_date'])
orders_with_customers = orders.merge(customers, on='customer_id', how='left')

print(orders_with_customers.head())
print()
print(orders_with_customers.dtypes)


This cell groups the data and orders the result so the learner can compare categories clearly.


In [ ]:
country_summary = (
    orders_with_customers
    .groupby('country')['amount']
    .agg(total_amount='sum', average_amount='mean', order_count='size')
    .sort_values('total_amount', ascending=False)
)

category_summary = (
    orders_with_customers
    .pivot_table(
        index='country',
        columns='category',
        values='amount',
        aggfunc='sum',
        fill_value=0,
    )
)

print(country_summary)
print()
print(category_summary)


This cell continues the worked example for the current section.


In [ ]:
clean_orders = (
    orders_with_customers
    .drop_duplicates()
    .assign(
        amount_eur=lambda df: df['amount'].round(2),
        order_month=lambda df: df['order_date'].dt.to_period('M').astype(str),
    )
)

print(clean_orders[['order_id', 'country', 'amount_eur', 'order_month']])


This cell continues the worked example for the current section.


In [ ]:
def format_amount(value: float) -> str:
    return f'{value:,.2f}'

amount_preview = clean_orders['amount'].head().apply(format_amount)

print('formatted amounts:')
print(amount_preview.tolist())

output_path = data_dir / 'orders_clean.csv'
clean_orders.to_csv(output_path, index=False)
print(output_path.resolve())


This cell continues the worked example for the current section.


In [ ]:
summary_path = artifacts_dir / 'pandas_deep_dive_summary.txt'
summary_path.write_text(
    'pandas deep dive summary\n'
    f'Rows: {len(clean_orders):,}\n'
    f'Countries: {", ".join(country_summary.index.tolist())}\n'
    f'Categories: {", ".join(sorted(clean_orders["category"].unique()))}\n'
)

print(summary_path.resolve())


<img src="https://hilpisch.com/tds_logo.png" width="20%" align="right">
